# Tutorial 8: Vision and Attention

This tutorial explores ACT-R's vision module, which simulates how humans allocate visual attention. We'll look at visual search, eye movements, and how attention affects what we can see and process in a scene.

## 1. Introduction to Visual Processing

ACT-R's vision module simulates human visual attention with limited capacity and serial search. The visual system has two key components: the visual location buffer (tracking WHERE things are) and the visual buffer (identifying WHAT things are).

In [ ]:
import pyactr as actr
import numpy as np

vision_model = actr.ACTRModel()

actr.chunktype("visual_object", "screen_x screen_y value color size")
actr.chunktype("visual_location", "screen_x screen_y color size")
actr.chunktype("visual_search_goal", "state target_color found_x found_y")

visual_objects = [
    (100, 100, "A", "red", "large"),
    (200, 100, "B", "blue", "small"),
    (150, 200, "C", "red", "small"),
    (250, 200, "D", "green", "large")
]

# In real ACT-R these would be in the environment, here we simulate with declarative memory
for x, y, val, color, size in visual_objects:
    vision_model.decmem.add(
        actr.makechunk(
            chunktype="visual_object",
            screen_x=str(x),
            screen_y=str(y),
            value=val,
            color=color,
            size=size
        )
    )

print("Visual scene created with 4 objects")
print("Objects vary in color, size, and location")
print("Model will search for specific features")

## 2. Visual Search Model

Visual search experiments distinguish between feature search (finding a red item among blue ones) and conjunction search (finding a red X among red Os and blue Xs). Feature search produces "pop-out" effects with constant search times, while conjunction search times increase linearly with the number of distractors.

In [ ]:
import pyactr as actr

search_model = actr.ACTRModel()

actr.chunktype("visual_target", "color shape")
actr.chunktype("search_goal", "state target_color looking_at")

search_model.productionstring(
    name="find_color",
    string='''
    =g>
        isa search_goal
        state searching
        target_color =color
    ==>
    =g>
        state locating
    +visual_location>
        isa visual_location
        color =color
        '''
)

search_model.productionstring(
    name="attend_location",
    string='''
    =g>
        isa search_goal
        state locating
    =visual_location>
        isa visual_location
        screen_x =x
        screen_y =y
    ==>
    =g>
        state attending
    +visual>
        isa move_attention
        screen_pos =visual_location
'''
)

search_model.productionstring(
    name="identify_object",
    string='''
    =g>
        isa search_goal
        state attending
    =visual>
        isa visual_object
        value =val
        color =color
    ==>
    =g>
        state found
        looking_at =val
'''
)

def search_time(target_present, set_size, feature_search=True):
    """Calculate visual search time"""
    if feature_search:
        return 0.5 if target_present else 0.5
    else:
        slope = 0.05 if target_present else 0.1
        return 0.5 + slope * set_size

print("Feature search (red among blue):")
for size in [4, 8, 16]:
    print(f"  Set size {size}: {search_time(True, size):.2f}s")

print("\nConjunction search (red X among red O and blue X):")
for size in [4, 8, 16]:
    print(f"  Set size {size}: {search_time(True, size, False):.2f}s")

## 3. Eye Movement Control

ACT-R models saccadic eye movements with timing based on the distance traveled. The time for a saccade includes a base time plus an increment proportional to the visual angle.

In [ ]:
import pyactr as actr
import math

eye_model = actr.ACTRModel()

eye_model.model_parameters["eye_saccade_rate"] = 0.002
eye_model.model_parameters["eye_saccade_base_time"] = 0.02

actr.chunktype("fixation_goal", "state current_fix next_x next_y")
actr.chunktype("fixation_sequence", "position x y next_pos")

fixations = [(100, 200), (180, 200), (260, 200), (340, 200)]
for i, (x, y) in enumerate(fixations):
    eye_model.decmem.add(
        actr.makechunk(
            chunktype="fixation_sequence",
            position=str(i),
            x=str(x),
            y=str(y),
            next_pos=str(i+1) if i < len(fixations)-1 else "done"
        )
    )

eye_model.productionstring(
    name="plan_saccade",
    string='''
    =g>
        isa fixation_goal
        state ready
        current_fix =pos
    ==>
    =g>
        state planning
    +retrieval>
        isa fixation_sequence
        position =pos
'''
)

def saccade_time(x1, y1, x2, y2, rate=0.002, base=0.02):
    distance = math.sqrt((x2-x1)**2 + (y2-y1)**2)
    visual_degrees = distance / 30
    return base + rate * visual_degrees

print("Reading saccade pattern:")
total_time = 0
for i in range(len(fixations)-1):
    x1, y1 = fixations[i]
    x2, y2 = fixations[i+1]
    time = saccade_time(x1, y1, x2, y2)
    total_time += time
    print(f"  Fixation {i} to {i+1}: {time*1000:.1f}ms")

print(f"\nTotal saccade time: {total_time*1000:.1f}ms")
print("Plus ~250ms per fixation for processing")

## 4. Visual Attention and the Spotlight

Visual attention in ACT-R follows a spotlight metaphor: detailed processing is only possible at the center of fixation, with decreasing acuity toward the periphery.

In [ ]:
import pyactr as actr

attention_model = actr.ACTRModel()
attention_model.model_parameters["subsymbolic"] = True

actr.chunktype("visual_item", "location detail_level content distance")
actr.chunktype("attention_goal", "focus_x focus_y task")

items = [
    (200, 200, "target", 0),
    (220, 200, "near1", 20),
    (250, 200, "near2", 50),
    (300, 200, "far1", 100),
    (400, 200, "periph", 200)
]

for x, y, content, dist in items:
    if dist == 0:
        detail = "high"
    elif dist < 50:
        detail = "medium"
    elif dist < 150:
        detail = "low"
    else:
        detail = "minimal"
    
    attention_model.decmem.add(
        actr.makechunk(
            chunktype="visual_item",
            location=f"{x},{y}",
            detail_level=detail,
            content=content,
            distance=str(dist)
        )
    )

attention_model.productionstring(
    name="process_detailed",
    string='''
    =g>
        isa attention_goal
        task identify_detailed
    =visual>
        isa visual_item
        detail_level high
        content =item
    ==>
    =g>
        task done
'''
)

attention_model.productionstring(
    name="detect_peripheral",
    string='''
    =g>
        isa attention_goal
        task detect_motion
    =visual_location>
        isa visual_item
        detail_level minimal
    ==>
    =g>
        task shift_attention
'''
)

print("Attention spotlight model created")
print("\nDetail levels by distance from fixation:")
print("  0 pixels: High detail (can read text)")
print("  <50 pixels: Medium detail (can identify objects)")
print("  <150 pixels: Low detail (basic features only)")
print("  >150 pixels: Minimal (motion/large changes only)")

## 5. Reading with Regressions

During reading, the eyes don't always move forward. Regressions (backward saccades) occur when the reader encounters difficulty, such as an unexpected word or complex syntax.

In [ ]:
import pyactr as actr

reading_model = actr.ACTRModel()
reading_model.model_parameters["subsymbolic"] = True

actr.chunktype("word", "position text frequency predictability")
actr.chunktype("reading_goal", "state current_word understanding")

sentence = [
    (0, "The", "high", "high"),
    (1, "quick", "high", "medium"),
    (2, "brown", "high", "high"),
    (3, "fox", "medium", "high"),
    (4, "jumps", "medium", "medium"),
    (5, "over", "high", "high"),
    (6, "the", "high", "high"),
    (7, "lazy", "medium", "low"),
    (8, "dog", "high", "high")
]

for pos, text, freq, pred in sentence:
    reading_model.decmem.add(
        actr.makechunk(
            chunktype="word",
            position=str(pos),
            text=text,
            frequency=freq,
            predictability=pred
        )
    )

reading_model.productionstring(
    name="read_forward",
    string='''
    =g>
        isa reading_goal
        state reading
        current_word =pos
        understanding good
    ==>
    =g>
        current_word =pos
    +visual>
        isa visual_location
        screen_x =pos
'''
)

reading_model.productionstring(
    name="regress_confused",
    string='''
    =g>
        isa reading_goal
        state reading
        current_word =pos
        understanding poor
    ==>
    =g>
        current_word =pos
        state rereading
    +visual>
        isa visual_location
        screen_x =pos
'''
)

reading_model.productionstring(
    name="skip_predictable",
    string='''
    =g>
        isa reading_goal
        state reading
        current_word =pos
    =visual>
        isa word
        position =pos
        predictability high
        frequency high
    ==>
    =g>
        current_word =pos
'''
)

print("Reading model with eye movement patterns:")
print("- Forward saccades (normal reading)")
print("- Regressions (when confused)")
print("- Word skipping (predictable words)")
print("\n'lazy' has low predictability - likely to cause regression!")

## 6. Visual Attention in Complex Scenes

When viewing complex scenes like websites, attention is driven by both bottom-up factors (visual salience) and top-down factors (task goals). Eye tracking studies show systematic patterns like the F-pattern in text-heavy layouts.

In [ ]:
import pyactr as actr

scene_model = actr.ACTRModel()

actr.chunktype("page_element", "type location salience content")
actr.chunktype("scan_goal", "task priority_target scanned")

elements = [
    ("header", "top", 5, "Company Logo"),
    ("nav", "top", 3, "Navigation Menu"),
    ("hero", "center", 10, "Big Sale Banner"),
    ("content", "center", 4, "Product List"),
    ("sidebar", "right", 6, "Special Offer"),
    ("footer", "bottom", 2, "Contact Info")
]

for elem_type, loc, sal, content in elements:
    scene_model.decmem.add(
        actr.makechunk(
            chunktype="page_element",
            type=elem_type,
            location=loc,
            salience=str(sal),
            content=content
        )
    )

# Bottom-up attention driven by salience
scene_model.productionstring(
    name="attend_salient",
    string='''
    =g>
        isa scan_goal
        task free_viewing
    ==>
    =g>
        task scanning
    +visual_location>
        isa page_element
        salience highest
        '''
)

# Top-down attention driven by task
scene_model.productionstring(
    name="find_target",
    string='''
    =g>
        isa scan_goal
        task search
        priority_target =target
    ==>
    +visual_location>
        isa page_element
        type =target
'''
)

scene_model.productionstring(
    name="f_pattern_scan",
    string='''
    =g>
        isa scan_goal
        task read_page
        scanned =area
    ==>
    =g>
        scanned next
    +visual_location>
        isa scan_sequence
        pattern f_shaped
        after =area
'''
)

print("Scene scanning model demonstrates:")
print("\n1. Bottom-up attention (salience-driven):")
print("   Hero banner (salience=10) viewed first")
print("   Then sidebar offer (salience=6)")
print("\n2. Top-down attention (task-driven):")
print("   Looking for nav? Goes straight to navigation")
print("\n3. F-pattern reading:")
print("   Top horizontal, down left side, occasional right scan")

## 7. Change Blindness

Change blindness demonstrates how unattended information is poorly encoded. People often fail to notice even large changes to a scene if those changes occur in unattended regions.

In [ ]:
import pyactr as actr

cb_model = actr.ACTRModel()
cb_model.model_parameters["subsymbolic"] = True

actr.chunktype("scene_object", "id location features attended encoding_strength")
actr.chunktype("change_detection", "state focus comparison")

scene_before = [
    ("person1", "left", "red_shirt", False, 0.3),
    ("person2", "center", "blue_shirt", True, 0.9),
    ("tree", "right", "green", False, 0.2),
    ("car", "background", "white", False, 0.1)
]

scene_after = [
    ("person1", "left", "yellow_shirt", False, 0.3),
    ("person2", "center", "blue_shirt", True, 0.9),
    ("tree", "right", "green", False, 0.2),
    ("car", "background", "white", False, 0.1)
]

cb_model.productionstring(
    name="detect_change",
    string='''
    =g>
        isa change_detection
        state comparing
    =retrieval>
        isa scene_object
        id =id
        features =old_features
        encoding_strength =strength!>0.5
    =visual>
        isa scene_object
        id =id
        features =new_features
        features ~=old_features
    ==>
    =g>
        state detected
'''
)

cb_model.productionstring(
    name="miss_change",
    string='''
    =g>
        isa change_detection
        state comparing
    =retrieval>
        isa scene_object
        encoding_strength =strength!<0.5
    ==>
    =g>
        state continue_search
'''
)

print("Change blindness demonstration:")
print("\nInitial encoding strengths:")
for obj, loc, feat, att, strength in scene_before:
    print(f"  {obj}: {strength} {'(attended)' if att else ''}")

print("\nPerson1's shirt changed from red to yellow!")
print("But encoding strength was only 0.3 (not attended)")
print("Change likely to be missed (change blindness)")
print("\nPerson2 well-encoded (0.9) - changes would be detected")

## Exercises

1. Create models for feature search vs conjunction search and compare times across different set sizes.

2. Model different reading strategies (skimming vs careful reading) with different eye movement patterns.

3. Model how sudden onset or motion captures attention even during focused tasks.

4. Model rapid scene categorization (indoor/outdoor) that occurs within 100ms.

5. Model tracking 3-4 moving objects simultaneously and predict when tracking fails.

The vision module provides a foundation for modeling how people interact with visual interfaces and environments. Combined with other ACT-R modules, you can build models that predict eye movements, attention allocation, and visual search performance in realistic tasks.